# Emotional Neural Network (ENN) - sustainable e-commerce delivery preferences

Code accompanying the paper *Predicting Sustainable E-Commerce Delivery Preferences Using Emotional Neural Networks: A Smart City Logistics Perspective*.

The notebook runs the repeated nested cross-validation for one input set (MENN1–MENN3) and one target (binary or multiclass).

Change `INPUTS` and `TARGET_TYPE` in Section 2 to reproduce the other configurations. The survey data are not publicly available.

## 1. Setup

In [ ]:
!pip install -q numba openpyxl

In [ ]:
import os, random
from itertools import product
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy import stats
from numba import njit
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score, precision_recall_curve, auc, brier_score_loss)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 2. Data and input sets

In [ ]:
# Column names of the (Serbian-language) survey file and English labels
TARGET = 'Da li biste platili više za ekološku opciju dostave, ukoliko bi Vam prodavac za to ponudio bodove lojalnosti koje možete zameniti za popuste ili donacije u humanitarne svrhe?'
# 1 = No, 2 = It depends on the circumstances, 3 = Yes

COLUMNS = [
    # Demographic
    ('Pol:', 'Gender'),
    ('Starosna grupa:', 'Age group'),
    ('Stečeni obrazovni nivo:\n','Educational level'),
    ('Grad:', 'City'), # City size: 1 = small, 2 = medium, 3 = large
    ('Prosečan mesečni prihod domaćinstva:', 'Average monthly household income'),

    # Environmental
    ('Koliko Vam je važno da proizvod bude upakovan u ekološki prihvatljivu ambalažu (npr. prirodni, reciklirani ili biorazgradivi materijali)?',
     'Ecological packaging'),
    ('Da li prisustvo ekoloških sertifikata / oznaka na proizvodima ili pakovanju utiču na vašu odluku o onlajn kupovini?',
     'Ecological certificates'),
    ('Prodavac vas informiše da će zasaditi drvo u širem regionu kupca za svaku 10. kupovinu (kupac dobija QR lokacije zasađenog drveta) ako izaberete opciju isporuke koja traje 2-3 puta duže od uobičajene (npr. 5 dana umesto 2 dana). Vaša reakcija je:',
     'Longer delivery time for planting trees'),
    ('Prodavac vas informiše da se odabirom opcije "Brza dostava" negativno utiče na životnu sredinu (povećava emisije CO2) u odnosu na opciju „Spora dostava“. Kako biste reagovali na ovu informaciju?',
     'Slow delivery to reduce CO2 emissions'),
    ('Da li biste češće koristili opciju preuzimanja proizvoda iz paketomata ili odabranog mesta za preuzimanje (prodavnica, pošta, benzinska stanica itd.) nego kućnu dostavu, ako bi znali da je to ekološki povoljnije?',
     'Parcel machines instead of home delivery'),
    ('Da li bi prihvatili da isporuka pošiljke bude izvršena tokom noći, jer bi se tako smanjile saobraćajne gužve tokom dana i ukupne emisije štetnih gasova?',
     'Night delivery to reduce emissions'),
    ('U kojoj meri mislite da trgovci manipulišu ekološkim temama da bi povećali cenu ili vreme Vaše isporuke?',
     'Manipulation of environmental issues'),
    ('Koliko biste bili spremni da produžite vreme čekanja na povrat proizvoda, da bi proces bio ekološki povoljniji?',
     'Longer returns for ecology'),

    # Economic
    ('Da li biste radije preuzeli paket iz paketomata ili sa mesta za lično preuzimanje ako bi to smanjilo cenu dostave?',
     'Parcel locker or CDP to reduce price'),
    ('Ako Vam je ponuđeno, da li kupujete veće količine i različite proizvode zajedno da bi dobili besplatnu dostavu?',
     'Larger purchases for free delivery'),
    ('Procenite važnost sledećih faktora prilikom online kupovine? - b) Cena isporuke',
     'Delivery price'),

    # Social
    ('Koliko je verovatno da biste proizvod kupili od prodavca kod kojeg je cena 5-10% viša, ako saznate da prodavac: - Promoviše raznolikost i inkluziju u zapošljavanju (npr. zapošljava osobe različitih nacionalnosti, religija, polova, osobe sa invaliditetom)',
     'Seller promotes inclusion'),
    ('Koliko je verovatno da biste proizvod kupili od prodavca kod kojeg je cena 5-10% viša, ako saznate da prodavac: - Poštuje propise i etičke principe u odnosu sa radnicima koji vrše isporuku (prijavljen radni staž i zdravstvena zaštita radnika)',
     'Seller respects ethical principles'),
    ('Koliko je verovatno da biste proizvod kupili od prodavca kod kojeg je cena 5-10% viša, ako saznate da prodavac: - Brine o široj zajednici u vašem gradu (doniranje vrtićima, staračkim domovima, školama, volonterske aktivnosti itd.)',
     'Seller contributes to local community'),
    ('U kojoj meri  bi Vas motivisalo ukoliko bi Vam e-trgovina ponudila digitalni bedž ili status (npr. ‘Zeleni kupac’) ako redovno birate održive opcije, što bi drugi korisnici mogli da vide (rang liste najboljih kupaca, takmičenja)?',
     'Green buyer status'),
]

ALL = [c for c, _ in COLUMNS]
LABELS_EN = dict(COLUMNS)
DEMOGRAPHIC = ALL[:5]  # demographic (5)
ATTITUDINAL = ALL[5:]  # environmental (8), economic (3), social (4)
# three items conceptually closest to the outcome (sensitivity analysis, response to Reviewer 2)
CLOSEST_ITEMS = [ALL[8], ALL[13], ALL[18]]

INPUT_SETS = {
    'MENN1': DEMOGRAPHIC,
    'MENN2': ATTITUDINAL,
    'MENN3': ALL,
    'MENN3_without3': [c for c in ALL if c not in CLOSEST_ITEMS],
}

In [ ]:
DATA_FILE = "survey.xlsx"      # not publicly available
INPUTS = "MENN1"               # 'MENN1', 'MENN2', 'MENN3', 'MENN3_without3'
TARGET_TYPE = "binary"         # 'binary', 'binary_alt', 'multiclass'

df = pd.read_excel(DATA_FILE)
X = df[INPUT_SETS[INPUTS]]
y3 = df[TARGET]                                      # 1 = No, 2 = It depends, 3 = Yes
if TARGET_TYPE == "binary":
    y, N_CLASSES = (y3 == 3).astype(int), 2          # Yes vs. No + It depends
elif TARGET_TYPE == "binary_alt":
    y, N_CLASSES = (y3 != 1).astype(int), 2          # Yes + It depends vs. No (sensitivity analysis)
else:
    y, N_CLASSES = y3.replace({1: 0, 2: 1, 3: 2}), 3

print(X.shape)
print(y.value_counts().sort_index())
print(X.isna().sum()[X.isna().sum() > 0])

## 3. ENN model

In [ ]:
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


class EANN:
    """
    Single emotional neural network.
    Parameters:
      - n_hidden - number of hidden neurons
      - n_hormones - number of hormones (K)
      - lr - base learning rate (step size eta = lr * mu)
      - momentum - momentum alpha for the conventional weights and biases
      - epochs - number of training epochs (online learning)
      - seed - random seed for initialisation and sample order
    """
    use_hormones = True

    def __init__(self, n_hidden=16, n_hormones=2, lr=0.05, momentum=0.8,
                 epochs=100, seed=42):
        self.n_hidden, self.n_hormones = n_hidden, n_hormones
        self.lr, self.momentum, self.epochs, self.seed = lr, momentum, epochs, seed

    # Initialisation
    def _init_params(self, n_in, n_out):
        rng = np.random.default_rng(self.seed)
        s1, s2 = np.sqrt(1.0 / n_in), np.sqrt(1.0 / self.n_hidden)
        self.W1 = rng.normal(0, s1, size=(n_in, self.n_hidden))           # input -> hidden
        self.b1 = np.zeros(self.n_hidden)
        self.W2 = rng.normal(0, s2, size=(self.n_hidden, n_out))          # hidden -> output
        self.b2 = np.zeros(n_out)
        self.C1 = rng.normal(0, s1, size=(self.n_hidden, self.n_hormones))  # hormonal weights, hidden
        self.C2 = rng.normal(0, s2, size=(n_out, self.n_hormones))          # hormonal weights, output
        self.G = rng.uniform(0.0, 0.5, size=(self.n_hidden, self.n_hormones))  # glandity (fixed)

    # Forward pass
    def _forward(self, x, H):
        h = _sigmoid(x @ self.W1 + self.b1 + self.C1 @ H)    # Eq. 8, hidden layer (additive hormones)
        o = _sigmoid(h @ self.W2 + self.b2 + self.C2 @ H)    # Eq. 8, output layer
        H_new = (self.G.T @ h) / self.n_hidden               # Eq. 9-10, normalised by n_hidden
        return h, o, H_new

    # Training: fast numba implementation
    def fit(self, X, Y, sample_weight=None):
        X = np.ascontiguousarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        if Y.ndim == 1:
            Y = Y.reshape(-1, 1)
        Y = np.ascontiguousarray(Y)
        self._init_params(X.shape[1], Y.shape[1])
        w = np.ones(X.shape[0]) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        rng = np.random.default_rng(self.seed)               # random sample order in each epoch
        idx = np.arange(X.shape[0])
        orders = np.empty((self.epochs, X.shape[0]), dtype=np.int64)
        for e in range(self.epochs):
            rng.shuffle(idx); orders[e] = idx
        # initial hormone level = mean of the training inputs (zero for the ablated network)
        H0 = (np.full(self.n_hormones, float(X.mean())) if self.use_hormones
              else np.zeros(self.n_hormones))
        self.H_final = _embp_train(X, Y, w, orders, self.W1, self.b1, self.W2, self.b2,
                                   self.C1, self.C2, self.G, H0,
                                   float(self.lr), float(self.momentum))
        return self

    # Prediction: hormone level fixed at its final training value
    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        H = self.H_final
        h = _sigmoid(X @ self.W1 + self.b1 + self.C1 @ H)
        return _sigmoid(h @ self.W2 + self.b2 + self.C2 @ H)


class EANNNoHormones(EANN):
    """Ablated ENN: same architecture, initialisation and learning, hormones switched off."""
    use_hormones = False

    def _init_params(self, n_in, n_out):
        super()._init_params(n_in, n_out)
        self.C1[:] = 0.0; self.C2[:] = 0.0; self.G[:] = 0.0

In [ ]:
@njit
def _sig_nb(z):
    out = np.empty_like(z)
    for j in range(z.shape[0]):
        v = min(max(z[j], -30.0), 30.0)
        out[j] = 1.0 / (1.0 + np.exp(-v))
    return out

@njit
def _embp_train(X, Y, w, orders, W1, b1, W2, b2, C1, C2, G, H, lr, mom):
    """Emotional backpropagation (online). Arrays are updated in place. Returns the final hormone vector."""
    n_h = W1.shape[1]
    dW1 = np.zeros_like(W1); db1 = np.zeros_like(b1)
    dW2 = np.zeros_like(W2); db2 = np.zeros_like(b2)
    dC1 = np.zeros_like(C1); dC2 = np.zeros_like(C2)
    pat_avg = np.empty(X.shape[0])
    for i in range(X.shape[0]):
        pat_avg[i] = X[i].mean()                    # mean of the input pattern (Y_avg in [7])
    mu0 = -1.0                                      # mean anxiety of the first epoch (set after epoch 1)
    for ep in range(orders.shape[0]):
        mu_sum = 0.0
        for q in range(orders.shape[1]):
            i = orders[ep, q]
            x = X[i]; t = Y[i]
            h = _sig_nb(x @ W1 + b1 + C1 @ H)                 # Eq. 8 / B1
            o = _sig_nb(h @ W2 + b2 + C2 @ H)                 # Eq. 8 / B1
            H_new = (G.T @ h) / n_h                           # Eq. 9-10 / B2
            err = t - o
            mu = pat_avg[i] + np.mean(np.abs(err))            # Eq. 14 / B3: anxiety
            k = 0.0 if mu0 < 0 else max(mu0 - mu, 0.0)        # Eq. 15 / B3: confidence
            mu_sum += mu
            d_o = w[i] * err * o * (1.0 - o)                  # output error term (class-weighted)
            d_h = (W2 @ d_o) * h * (1.0 - h)                  # hidden error term
            eta = lr * mu                                     # scaled emotional learning step
            dW2 = eta * np.outer(h, d_o) + mom * dW2; W2 += dW2   # Eq. 11 / B4
            db2 = eta * d_o + mom * db2;              b2 += db2   # Eq. 12 / B4
            dC2 = eta * np.outer(d_o, H) + k * dC2;   C2 += dC2   # Eq. 13 / B4
            dW1 = eta * np.outer(x, d_h) + mom * dW1; W1 += dW1   # Eq. 11 / B5
            db1 = eta * d_h + mom * db1;              b1 += db1   # Eq. 12 / B5
            dC1 = eta * np.outer(d_h, H) + k * dC1;   C1 += dC1   # Eq. 13 / B5
            H = 0.8 * H + 0.2 * H_new                         # smoothed hormonal feedback
        if mu0 < 0:
            mu0 = mu_sum / orders.shape[1]
    return H

In [ ]:
SEEDS = [42, 43, 44, 45, 46]

class EANNEnsemble:
    """Seed ensemble: five networks with identical hyperparameters and different random seeds; the predicted probabilities are averaged."""
    base = EANN

    def __init__(self, **params):
        self.params = params

    def fit(self, X, Y, sample_weight=None):
        self.nets = [self.base(**self.params, seed=s).fit(X, Y, sample_weight=sample_weight) for s in SEEDS]
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.nets], axis=0)

class EANNNoHormonesEnsemble(EANNEnsemble):
    base = EANNNoHormones

## 4. Nested cross-validation

### 4.1 Preprocessing for the reference models
Median imputation and standardisation, fitted on the training part only (the ENN uses min–max scaling).

In [ ]:
def prep_fold(X_tr, X_val):
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(imputer.fit_transform(X_tr))
    X_val = scaler.transform(imputer.transform(X_val))
    return X_tr, X_val

### 4.2 Hyperparameter selection (inner cross-validation) and candidate values

In [ ]:
PARAMS = defaultdict(list)    # selected hyperparameters per outer fold, for each model

def param_grid(d):
    return [dict(zip(d.keys(), v)) for v in product(*d.values())]

def select_params(train_predict, grid, X_tr, y_tr, n_inner=3):
    # Inner CV ONLY on the training part of the outer fold: for every combination compute the mean macro F1 and return the best combination.
    y_tr = np.asarray(y_tr)
    inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=SEED)
    best_p, best_f1 = None, -1
    for p in param_grid(grid):
        f1s = []
        for a, b in inner.split(X_tr, y_tr):
            pred = train_predict(p, X_tr.iloc[a], y_tr[a], X_tr.iloc[b])
            f1s.append(f1_score(y_tr[b], pred, average='macro'))
        if np.mean(f1s) > best_f1:
            best_p, best_f1 = p, np.mean(f1s)
    print(f"   selected: {best_p}  (inner F1 = {best_f1:.3f})")
    return best_p

# Candidate values for each model
GRID_ENN    = {'n_hidden': [16, 48], 'n_hormones': [2, 4], 'lr': [0.01, 0.02, 0.05, 0.3], 'epochs': [25, 50, 100, 200, 400], 'momentum': [0.5, 0.8]}
GRID_ENN_NH = {'n_hidden': [16, 48], 'lr': [0.01, 0.02, 0.05, 0.3], 'epochs': [25, 50, 100, 200, 400], 'momentum': [0.5, 0.8]}
GRID_LR     = {'C': [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]}
GRID_RF     = {'max_depth': [3, 5, 8, None], 'min_samples_leaf': [1, 3, 5, 10], 'max_features': ['sqrt', 0.5, 1.0]}
GRID_DT     = {'max_depth': [2, 3, 4, 5, None], 'min_samples_leaf': [1, 5, 10, 20]}

def is_binary(y):
    return len(np.unique(y)) == 2

def tp_enn(p, X_a, y_a, X_b, Model=EANN):
    imp = SimpleImputer(strategy='median'); sc = MinMaxScaler()
    Z_a = sc.fit_transform(imp.fit_transform(X_a))
    Z_b = np.clip(sc.transform(imp.transform(X_b)), 0, 1)
    sw = compute_sample_weight('balanced', y_a)
    if is_binary(y_a):
        m = Model(**p, seed=SEED).fit(Z_a, y_a, sample_weight=sw)
        return (m.predict_proba(Z_b).ravel() >= 0.5).astype(int)
    m = Model(**p, seed=SEED).fit(Z_a, np.eye(N_CLASSES)[y_a], sample_weight=sw)
    return m.predict_proba(Z_b).argmax(axis=1)

def tp_enn_nh(p, X_a, y_a, X_b):
    return tp_enn(p, X_a, y_a, X_b, Model=EANNNoHormones)

def tp_lr(p, X_a, y_a, X_b):
    Z_a, Z_b = prep_fold(X_a, X_b)
    return LogisticRegression(C=p['C'], class_weight='balanced', max_iter=5000).fit(Z_a, y_a).predict(Z_b)

def tp_rf(p, X_a, y_a, X_b):
    Z_a, Z_b = prep_fold(X_a, X_b)
    return RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, **p).fit(Z_a, y_a).predict(Z_b)

def tp_dt(p, X_a, y_a, X_b):
    Z_a, Z_b = prep_fold(X_a, X_b)
    return DecisionTreeClassifier(class_weight='balanced', random_state=SEED, **p).fit(Z_a, y_a).predict(Z_b)

### 4.3 Final model of an outer fold: predicted probabilities and metrics

In [ ]:
def probabilities(name, p, X_tr, y_tr, X_te):
    # Trains the model with the selected parameters on the outer training part and returns predicted probabilities for the outer test part.
    y_tr = np.asarray(y_tr)
    if name in ('ENN', 'ENN_no_hormones'):
        # ENN: median + min-max (0-1), class weights, ensemble of five networks
        imp = SimpleImputer(strategy='median').fit(X_tr)
        sc = MinMaxScaler().fit(imp.transform(X_tr))
        A = sc.transform(imp.transform(X_tr))
        B = sc.transform(imp.transform(X_te))
        if name == 'ENN_no_hormones':
            B = np.clip(B, 0, 1)
        Model = EANNEnsemble if name == 'ENN' else EANNNoHormonesEnsemble
        Y = y_tr if N_CLASSES == 2 else np.eye(N_CLASSES)[y_tr]
        P = Model(**p).fit(A, Y, sample_weight=compute_sample_weight('balanced', y_tr)).predict_proba(B)
        return P.ravel() if N_CLASSES == 2 else P

    A, B = prep_fold(X_tr, X_te)                    # median + standardisation
    if name == 'LR':
        m = LogisticRegression(C=p['C'], class_weight='balanced', max_iter=5000)
    elif name == 'DT':
        m = DecisionTreeClassifier(class_weight='balanced', random_state=SEED, **p)
    elif name == 'RF':
        m = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, **p)
    P = m.fit(A, y_tr).predict_proba(B)
    return P[:, 1] if N_CLASSES == 2 else P

def metrics(y_te, P):
    y_te = np.asarray(y_te)
    if N_CLASSES == 2:
        pred = (P > 0.5).astype(int)                # decision threshold 0.5
        prec, rec, _ = precision_recall_curve(y_te, P)
        return {'Accuracy': accuracy_score(y_te, pred), 'F1_macro': f1_score(y_te, pred, average='macro'),
                'ROC_AUC': roc_auc_score(y_te, P), 'PR_AUC': auc(rec, prec),
                'Brier': brier_score_loss(y_te, P)}
    pred = P.argmax(axis=1)
    Pn = P / P.sum(axis=1, keepdims=True)            # normalisation (ENN outputs need not sum to one)
    return {'Accuracy': accuracy_score(y_te, pred), 'F1_macro': f1_score(y_te, pred, average='macro'),
            'ROC_AUC': roc_auc_score(y_te, Pn, multi_class='ovr', average='macro'),
            'Brier': np.mean([brier_score_loss((y_te == c).astype(int), Pn[:, c]) for c in range(N_CLASSES)])}

### 4.4 Outer loop: stratified 5-fold cross-validation repeated 5 times (25 folds)

In [ ]:
skf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
RESULTS = {}

def run_model(name, train_predict, grid):
    rows = []
    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        print(f"{name} - fold {fold}")
        best_p = select_params(train_predict, grid, X.iloc[tr], y.iloc[tr])   # training part only
        PARAMS[name].append(best_p)
        P = probabilities(name, best_p, X.iloc[tr], y.iloc[tr], X.iloc[te])
        rows.append({'fold': fold, **metrics(y.iloc[te], P)})
    RESULTS[name] = pd.DataFrame(rows).set_index('fold')
    display(RESULTS[name].agg(['mean', 'std']))

### 4.5 Run the models

In [ ]:
run_model('ENN', tp_enn, GRID_ENN)

In [ ]:
run_model('ENN_no_hormones', tp_enn_nh, GRID_ENN_NH)

In [ ]:
run_model('LR', tp_lr, GRID_LR)

In [ ]:
run_model('DT', tp_dt, GRID_DT)

In [ ]:
run_model('RF', tp_rf, GRID_RF)

In [ ]:
results = pd.concat({k: v.reset_index() for k, v in RESULTS.items()}, names=['model']).reset_index(level=0)
results.to_csv(f"results_{INPUTS}_{TARGET_TYPE}.csv", index=False)

## 5. Results: mean ± SD [95% CI] and paired tests

### 5.1 Statistical helpers (Nadeau & Bengio corrected CI and t-test, Wilcoxon, Holm)

In [ ]:
def corrected_ci(v, n_total, n_splits=5, level=0.95):
    v = np.asarray(v); J = len(v)
    n_test = n_total / n_splits; n_train = n_total - n_test
    h = stats.t.ppf(0.5 + level / 2, J - 1) * np.sqrt((1 / J + n_test / n_train) * v.var(ddof=1))
    return v.mean() - h, v.mean() + h

def paired_tests(a, b, n_total, n_splits=5):
    d = np.asarray(a) - np.asarray(b); J = len(d)
    if np.allclose(d, 0):
        return d.mean(), 1.0, 1.0
    n_test = n_total / n_splits; n_train = n_total - n_test
    t = d.mean() / np.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    return d.mean(), 2 * stats.t.sf(abs(t), J - 1), stats.wilcoxon(d).pvalue

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(run, 1.0)
    return adj

### 5.2 Summary table

In [ ]:
def summary(res, n_total):
    out = []
    for name, g in res.groupby('model', sort=False):
        row = {'model': name}
        for m in [c for c in g.columns if c not in ('model', 'fold', 'params')]:
            lo, hi = corrected_ci(g[m], n_total)
            row[m] = f"{g[m].mean():.3f} ± {g[m].std(ddof=1):.3f} [{lo:.3f}, {hi:.3f}]"
        out.append(row)
    return pd.DataFrame(out)

summary(results, len(df))

### 5.3 ENN vs. reference models

In [ ]:
# ENN vs. reference models: corrected resampled t-test and Wilcoxon test, Holm correction
enn = results[results.model == 'ENN'].sort_values('fold')['F1_macro'].values
tests = []
for name in ['ENN_no_hormones', 'LR', 'DT', 'RF']:
    other = results[results.model == name].sort_values('fold')['F1_macro'].values
    d, p_t, p_w = paired_tests(enn, other, len(df))
    tests.append({'comparison': f'ENN vs {name}', 'diff_F1': round(d, 4), 'p_corrected_t': p_t, 'p_wilcoxon': p_w})
tests = pd.DataFrame(tests)
tests['p_holm_t'] = holm(tests.p_corrected_t)
tests['p_holm_wilcoxon'] = holm(tests.p_wilcoxon)
tests